In [5]:
import torch
import matplotlib.pyplot as plt

from src.data.labels import load_labels
from src.data.split_data import split_patients
from src.data.dataset import RSNADataset
from src.data.dataloader import build_dataloader

from src.models.baseline_cnn import BaselineCNN
from src.training.losses import build_loss
from src.training.optimizers import build_optimizer
from src.training.train import train_one_epoch
from src.training.validate import validate

In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [7]:
from src.utils.config import TRAIN_LABELS, TRAIN_IMAGES

labels, stats = load_labels(TRAIN_LABELS)

print("Total patients:", len(labels))

Total patients: 26684


Patient-level split

In [8]:
train_ids, val_ids, test_ids = split_patients(
    labels=labels,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    random_seed=42,
)

print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

Train: 18678
Validation: 4002
Test: 4004


Create datasets

Remember: your RSNADataset expects:

patient_ids,
labels,
image_dir

In [9]:
train_dataset = RSNADataset(
    patient_ids=train_ids,
    labels=labels,
    image_dir=TRAIN_IMAGES,
)

val_dataset = RSNADataset(
    patient_ids=val_ids,
    labels=labels,
    image_dir=TRAIN_IMAGES,
)

test_dataset = RSNADataset(
    patient_ids=test_ids,
    labels=labels,
    image_dir=TRAIN_IMAGES,
)

Create DataLoaders
Use your existing build_dataloader().

In [10]:
train_loader = build_dataloader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
)

val_loader = build_dataloader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

test_loader = build_dataloader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 2335
Validation batches: 501
Test batches: 501


Create model

In [11]:
model = BaselineCNN().to(device)

print(model)

BaselineCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): AdaptiveAvgPool2d(output_size=(1, 1))
    (1): Flatten(start_dim=1, end_dim=-1)
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=1, bias=True)
    (5): Sigmoid()
  )
)


Create loss

In [13]:
criterion = build_loss()

print(criterion)

BCELoss()


Create optimizer

In [15]:
optimizer = build_optimizer(
    model,
    learning_rate =0.001,
)

print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


Training configuration

Because this is our first complete end-to-end run.

If everything works:
5 epochs → verify → then 10 epochs if needed

In [16]:
num_epochs = 5

train_losses = []
val_losses = []

Training loop

In [ ]:
for epoch in range(num_epochs):

    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    train_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
    )

    val_loss = validate(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=device,
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")

# 5 epochs took:
# ~130 minutes = ~26 minutes/epoch.

# And the important part is:
# Train: 18,678 images
# Batch size: 8
# ≈ 2,335 batches/epoch
# So we're doing roughly:
# 2,335 batches
# × 5 epochs
# = 11,675 batches
# And each batch is reading DICOM → preprocessing → transferring to GPU → CNN inference
    
# Epoch 1/5
# Batch [100/2335] Loss: 0.5694
# Batch [200/2335] Loss: 0.7328
# Batch [300/2335] Loss: 0.2251
# Batch [400/2335] Loss: 0.4330
# Batch [500/2335] Loss: 0.5826
# Batch [600/2335] Loss: 0.4230
# Batch [700/2335] Loss: 0.4082
# Batch [800/2335] Loss: 0.9642
# Batch [900/2335] Loss: 0.5800
# Batch [1000/2335] Loss: 0.2143
# Batch [1100/2335] Loss: 0.5515
# Batch [1200/2335] Loss: 0.4979
# Batch [1300/2335] Loss: 0.6958
# Batch [1400/2335] Loss: 0.2344
# Batch [1500/2335] Loss: 0.3920
# Batch [1600/2335] Loss: 0.5663
# Batch [1700/2335] Loss: 0.4243
# Batch [1800/2335] Loss: 0.2384
# Batch [1900/2335] Loss: 0.6407
# Batch [2000/2335] Loss: 0.5683
# Batch [2100/2335] Loss: 0.7484
# Batch [2200/2335] Loss: 0.7126
# Batch [2300/2335] Loss: 0.7149
# Train Loss: 0.5329
# Val Loss:   0.5186

# Epoch 2/5
# Batch [100/2335] Loss: 0.5536
# Batch [200/2335] Loss: 0.6586
# Batch [300/2335] Loss: 0.5784
# Batch [400/2335] Loss: 0.6025
# Batch [500/2335] Loss: 0.2239
# Batch [600/2335] Loss: 0.5502
# Batch [700/2335] Loss: 0.5640
# Batch [800/2335] Loss: 0.3856
# Batch [900/2335] Loss: 0.4214
# Batch [1000/2335] Loss: 0.5639
# Batch [1100/2335] Loss: 0.4426
# Batch [1200/2335] Loss: 0.2743
# Batch [1300/2335] Loss: 0.2786
# Batch [1400/2335] Loss: 0.1689
# Batch [1500/2335] Loss: 0.7159
# Batch [1600/2335] Loss: 0.5092
# Batch [1700/2335] Loss: 0.4540
# Batch [1800/2335] Loss: 0.3988
# Batch [1900/2335] Loss: 0.6863
# Batch [2000/2335] Loss: 0.5166
# Batch [2100/2335] Loss: 0.5904
# Batch [2200/2335] Loss: 0.4218
# Batch [2300/2335] Loss: 0.2247
# Train Loss: 0.5189
# Val Loss:   0.4971

# Epoch 3/5
# Batch [100/2335] Loss: 0.7667
# Batch [200/2335] Loss: 0.4482
# Batch [300/2335] Loss: 0.5974
# Batch [400/2335] Loss: 0.7399
# Batch [500/2335] Loss: 0.5791
# Batch [600/2335] Loss: 0.4776
# Batch [700/2335] Loss: 0.5043
# Batch [800/2335] Loss: 0.4670
# Batch [900/2335] Loss: 0.4666
# Batch [1000/2335] Loss: 0.4546
# Batch [1100/2335] Loss: 0.6310
# Batch [1200/2335] Loss: 0.3657
# Batch [1300/2335] Loss: 0.6243
# Batch [1400/2335] Loss: 0.4810
# Batch [1500/2335] Loss: 0.3845
# Batch [1600/2335] Loss: 0.1672
# Batch [1700/2335] Loss: 0.1928
# Batch [1800/2335] Loss: 0.4028
# Batch [1900/2335] Loss: 1.0283
# Batch [2000/2335] Loss: 0.9995
# Batch [2100/2335] Loss: 0.3516
# Batch [2200/2335] Loss: 0.5341
# Batch [2300/2335] Loss: 0.2734
# Train Loss: 0.4980
# Val Loss:   0.4790

# Epoch 4/5
# Batch [100/2335] Loss: 0.3592
# Batch [200/2335] Loss: 0.5307
# Batch [300/2335] Loss: 0.8904
# Batch [400/2335] Loss: 0.2570
# Batch [500/2335] Loss: 0.2427
# Batch [600/2335] Loss: 0.4044
# Batch [700/2335] Loss: 0.4371
# Batch [800/2335] Loss: 0.3039
# Batch [900/2335] Loss: 0.6497
# Batch [1000/2335] Loss: 0.1404
# Batch [1100/2335] Loss: 0.5150
# Batch [1200/2335] Loss: 0.5403
# Batch [1300/2335] Loss: 0.4941
# Batch [1400/2335] Loss: 0.4315
# Batch [1500/2335] Loss: 0.5728
# Batch [1600/2335] Loss: 0.5400
# Batch [1700/2335] Loss: 0.5899
# Batch [1800/2335] Loss: 0.4105
# Batch [1900/2335] Loss: 0.4653
# Batch [2000/2335] Loss: 0.5129
# Batch [2100/2335] Loss: 1.3204
# Batch [2200/2335] Loss: 0.7324
# Batch [2300/2335] Loss: 0.2872
# Train Loss: 0.4867
# Val Loss:   0.4775

# Epoch 5/5
# Batch [100/2335] Loss: 0.8091
# Batch [200/2335] Loss: 0.1844
# Batch [300/2335] Loss: 0.5575
# Batch [400/2335] Loss: 0.5155
# Batch [500/2335] Loss: 0.5174
# Batch [600/2335] Loss: 0.1416
# Batch [700/2335] Loss: 0.4765
# Batch [800/2335] Loss: 0.6761
# Batch [900/2335] Loss: 0.8197
# Batch [1000/2335] Loss: 0.2918
# Batch [1100/2335] Loss: 0.3196
# Batch [1200/2335] Loss: 0.6826
# Batch [1300/2335] Loss: 0.5764
# Batch [1400/2335] Loss: 0.5495
# Batch [1500/2335] Loss: 0.3356
# Batch [1600/2335] Loss: 0.3674
# Batch [1700/2335] Loss: 0.2482
# Batch [1800/2335] Loss: 0.2097
# Batch [1900/2335] Loss: 0.3629
# Batch [2000/2335] Loss: 0.5115
# Batch [2100/2335] Loss: 0.3741
# Batch [2200/2335] Loss: 0.4407
# Batch [2300/2335] Loss: 0.3742
# Train Loss: 0.4812
# Val Loss:   0.4828

In [20]:
from src.utils.checkpoint import save_checkpoint
from src.utils.config import MODEL_DIR

checkpoint_path = MODEL_DIR / "baseline_cnn_checkpoint.pth"

save_checkpoint(
    model=model,
    optimizer=optimizer,
    epoch=num_epochs,
    loss=val_losses[-1],
    path=checkpoint_path,
)

Checkpoint saved to: C:\Users\Ananth Narayan L S\Desktop\Projects\AstraMed_v1.0\models\baseline_cnn_checkpoint.pth


In [23]:
from src.models.baseline_cnn import BaselineCNN
from src.training.optimizers import build_optimizer
from src.utils.checkpoint import load_checkpoint

loaded_model = BaselineCNN().to(device)

loaded_optimizer = build_optimizer(
    loaded_model,
    learning_rate=0.001,
)

checkpoint = load_checkpoint(
    model=loaded_model,
    optimizer=loaded_optimizer,
    path=checkpoint_path,
    device=device,
)

print("Loaded epoch:", checkpoint["epoch"])
print("Loaded loss:", checkpoint["loss"])

Checkpoint loaded from: C:\Users\Ananth Narayan L S\Desktop\Projects\AstraMed_v1.0\models\baseline_cnn_checkpoint.pth
Loaded epoch: 5
Loaded loss: 0.48284427413548425


real metrics

In [24]:
model.eval()

all_targets = []
all_probabilities = []

with torch.no_grad():

    for batch in val_loader:

        images = batch["image"].to(device)

        outputs = model(images)

        probabilities = outputs.squeeze(1)

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )

        all_targets.extend(
            batch["target"].numpy()
        )

print("Number of validation samples:", len(all_targets))
print("Number of probabilities:", len(all_probabilities))

Number of validation samples: 4002
Number of probabilities: 4002


In [25]:
from src.evaluations.metrics import calculate_metrics

metrics = calculate_metrics(
    targets=all_targets,
    probabilities=all_probabilities,
)

In [26]:
print(f"Accuracy : {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall   : {metrics['recall']:.4f}")
print(f"F1 Score : {metrics['f1']:.4f}")
print(f"ROC-AUC  : {metrics['roc_auc']:.4f}")

print("\nConfusion Matrix:")
print(metrics["confusion_matrix"])

Accuracy : 0.7806
Precision: 0.3333
Recall   : 0.0034
F1 Score : 0.0068
ROC-AUC  : 0.7269

Confusion Matrix:
[[3121    6]
 [ 872    3]]


In [27]:
assert len(all_targets) == len(val_dataset), (
    f"Expected {len(val_dataset)} samples, "
    f"got {len(all_targets)}"
)